# Visium - SCA013

In [1]:
# TODO: update Visium reader in harpy

## Import packages

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import dask as da
import os
import gc
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import seaborn as sns
import h5py
import json
import gc

from dask_image.imread import imread
import spatialdata_plot

from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\xarray_schema\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [4]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize
from utils.qc import qc_metrics
from _visium import visium

## Create sdata

In [5]:
dataset_id = "SCA013"

path_outs = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium/{dataset_id}/COUNTS/{dataset_id}/{dataset_id}_reseq/outs'
counts_file = "raw_feature_bc_matrix.h5"
path_img = f'D:/2024-07-UTBenchmark-SpC/data/raw/Visium/{dataset_id}/IMAGES/{dataset_id}_HE_hires.tif'
output_path = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium/{dataset_id}/harpy/sdata.zarr'

pipeline_name = 'SpaceRanger'
species = 'mouse'
tissue_annotation_file = f'D:/2024-07-UTBenchmark-SpC/data/processed/Visium/{dataset_id}/tissue_annotation/tissue.geojson'

In [6]:
# Create SpatialData from Space Ranger output
sdata = visium(
    path = path_outs, 
    counts_file = counts_file,
    dataset_id = dataset_id,
    fullres_image_file = path_img,
    output = output_path
)
sdata

INFO     reading                                                                                                   
         D:\2024-07-UTBenchmark-SpC\data\raw\Visium\SCA013\COUNTS\SCA013\SCA013_reseq\outs\raw_feature_bc_matrix.h5


c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\models\models.py:1144: UserWarning: Converting `region_key: region` to categorical dtype.
  return convert_region_column_to_categorical(adata)
<tifffile.TiffFile 'SCA013_HE_hires.tif'> ImageJ series metadata invalid or corrupted file
c:\Users\julien.mortier\conda-envs\harpy\lib\site-packages\spatialdata\_core\_elements.py:105: UserWarning: Key `SCA013` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
2025-08-25 17:34:42,230 - harpy

INFO     The Zarr backing store has been changed from None the new file path:                                      
         D:\2024-07-UTBenchmark-SpC\data\processed\Visium\SCA013\harpy\sdata.zarr                                  


version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04
version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: D:\2024-07-UTBenchmark-SpC\data\processed\Visium\SCA013\harpy\sdata.zarr
├── Images
│     ├── 'SCA013_full_image': DataTree[cyx] (3, 11995, 14569), (3, 5997, 7284), (3, 2998, 3642), (3, 1499, 1821), (3, 749, 910)
│     ├── 'SCA013_hires_image': DataArray[cyx] (3, 1647, 2000)
│     └── 'SCA013_lowres_image': DataArray[cyx] (3, 494, 600)
├── Labels
│     └── 'SCA013_labels': DataArray[yx] (13860, 14162)
├── Shapes
│     └── 'SCA013': GeoDataFrame shape: (4970, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (4970, 32285)
with coordinate systems:
    ▸ 'SCA013', with elements:
        SCA013_full_image (Images), SCA013_hires_image (Images), SCA013_lowres_image (Images), SCA013_labels (Labels), SCA013 (Shapes)
    ▸ 'SCA013_downscaled_hires', with elements:
        SCA013_hires_image (Images), SCA013_labels (Labels), SCA013 (Shapes)
    ▸ 'SCA013_downscaled_lowres', with elements:
        SCA013_lowres_image (Images), SCA013_labels (Labe

## Specify plotting parameters

In [7]:
DPI = 200
MAX_DIM_FOR_PLOTS = 5000

HE_image = f'{dataset_id}_full_image'

best_scale = best_datatree_scale(sdata[HE_image], MAX_DIM_FOR_PLOTS)
print(f'best scale: {best_scale}')

figsize_factor = DPI
figsize = (sdata.images[HE_image][best_scale]['image'].shape[2]/figsize_factor, sdata.images[HE_image][best_scale]['image'].shape[1]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)
print(f'plots_path: {plots_path}')

best scale: scale2
figsize: (18.21, 14.99)
plots_path: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots


## Plot H&E

In [8]:
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "H&E"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'HE.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\HE.png


## Plot Space Ranger tissue annotation

In [9]:
# Check spots
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = f"{pipeline_name} - tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_labels(
    element=f"{dataset_id}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue', scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'{pipeline_name}_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)


INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\SpaceRanger_tissue_annotation.png


## Calculate pixel size

In [10]:
# Determine pixel size in microns
import tifffile

with tifffile.TiffFile(path_img) as tif:
    xres = tif.pages[0].tags["XResolution"].value  # (numerator, denominator)
    yres = tif.pages[0].tags["YResolution"].value
    unit = tif.pages[0].tags["ResolutionUnit"].value

    px_per_cm_x = xres[0] / xres[1]
    px_per_cm_y = yres[0] / yres[1]

    if unit.name == "CENTIMETER":
        microns_per_pixel_x = 10_000 / px_per_cm_x
        microns_per_pixel_y = 10_000 / px_per_cm_y
    elif unit.name == "INCH":
        microns_per_pixel_x = 25_400 / px_per_cm_x
        microns_per_pixel_y = 25_400 / px_per_cm_y
    else:
        raise ValueError(f"Unknown ResolutionUnit: {unit}")

if microns_per_pixel_x == microns_per_pixel_y:
    microns_per_pixel = microns_per_pixel_x
    print(f"Pixel size: {microns_per_pixel:.3f} µm")
else:
    print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")


Pixel size: 0.442 µm


## Create bounding box for total capture area

In [11]:
# Add bounding box for total capture area
union_geom = sdata.shapes[dataset_id].union_all()
bbox = union_geom.minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

In [12]:
# Check capture area bbox
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area bounding box"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=dataset_id, color='black', fill_alpha=0.5, outline_width=rel_fontsize(fig, 0.0012), outline_alpha=1, outline_color='black'
).pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='#1f77b4', fill_alpha=0.1, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'capture_area_bbox.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\capture_area_bbox.png


In [13]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

Area of capture area: 43.78 mm²


In [14]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

Dimensions of capture area bbox: 0.65 cm x 0.68 cm


## Add tissue annotation

In [15]:
# Add tissue annotation as shapes layer and as labels layer
gdf = gpd.read_file(tissue_annotation_file)
gdf = gdf.explode(index_parts=False) # This avoid problems with MultiPolygon geometries
gdf.index = gdf.index + 1 # Index 0 is not allowed

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gdf,
    output_layer = f'{dataset_id}_tissue',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

sdata = hp.im.rasterize(
    sdata,
    shapes_layer = f'{dataset_id}_tissue',
    output_layer = f'{dataset_id}_tissue_mask',
    chunks = 1024,
    overwrite = True,
)

2025-08-25 17:35:39,676 - harpy.image._manager - INFO - Writing results to layer 'SCA013_tissue_mask'


In [16]:
# Check tissue annotation
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_shapes(
    element=f'{dataset_id}_tissue', color='#1f77b4', fill_alpha=0.3, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\qupath_tissue_annotation.png


In [17]:
# Determine which spots are in tissue annotation
sdata = add_in_tissue(
    sdata, 
    table_layer = 'table', 
    tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
)

2025-08-25 17:35:57,258 - harpy.utils._io - WARNING - layer with name 'table' already exists. Overwriting...


In [18]:
# Check spots in tissue
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Tissue annotation spots"

sdata.pl.render_images(
    element=HE_image, scale=best_scale
).pl.render_labels(
    element=f"{dataset_id}_labels", fill_alpha=0.5, contour_px=5, outline_alpha=1, color='in_tissue_annotation', table_layer='table', scale=best_scale
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.02)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, f'qupath_tissue_annotation_spots.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\qupath_tissue_annotation_spots.png


In [19]:
tissue_positions = pd.read_csv(os.path.join(path_outs, 'spatial', 'tissue_positions.csv'))
tissue_positions

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,GTCACTTCCTTCTAGA-1,0,0,0,-1582.142537,13854.550780
1,TAGATGCGTGGTTGAA-1,0,1,1,-1376.988929,13744.447800
2,CACGGTCTCCTTACGA-1,0,0,2,-1590.199338,13628.279231
3,AGGATCCGTAAGTAAT-1,0,1,3,-1385.045730,13518.176252
4,ATAGCTGCGGATAAGA-1,0,0,4,-1575.641868,13402.335552
...,...,...,...,...,...,...
4987,GTCGAGACCGTCCGTT-1,0,77,123,13780.946920,146.158399
4988,TACGCCAGCACCGGAA-1,0,76,124,13579.043646,30.153765
4989,TGGCTGGAGTTATTGT-1,0,77,125,13772.890119,-80.113150
4990,ACAGTTGAGGCTAATG-1,0,76,126,13582.293981,-195.953849


In [20]:
# Total number of spots in capture area
print(f"Total number of spots in capture area: {len(tissue_positions)}")

Total number of spots in capture area: 4992


In [21]:
# Total number of spots in tissue
tissue_mask = sdata.labels[f'{dataset_id}_tissue_mask'].compute().data
mask_shape = tissue_mask.shape

tissue_positions["in_tissue_annotation"] = tissue_positions.apply(
    lambda row: assign_region((row["pxl_col_in_fullres"], row["pxl_row_in_fullres"]), tissue_mask, mask_shape), axis=1
)

total_n_spots_in_tissue = len(tissue_positions[tissue_positions['in_tissue_annotation'] == 1])

print(f"Total number of spots in tissue: {total_n_spots_in_tissue}")

Total number of spots in tissue: 1779


In [22]:
# Total number of spots in table
n_spots = len(sdata.tables['table'].obs)
print(f"Number of spots in table: {n_spots}")

Number of spots in table: 4970


In [24]:
# Number of spots in tissue (in table)
n_spots_in_tissue = len(sdata.tables['table'][sdata.tables['table'].obs['in_tissue_annotation'] == 1].obs)
print(f"Number of spots in tissue: {n_spots_in_tissue}")

Number of spots in tissue: 1779


In [26]:
# Area of tissue annotation that overlaps with capture area in mm²
tissue_gdf = sdata.shapes[f"{dataset_id}_tissue"].unary_union
capture_gdf = sdata.shapes[f"{dataset_id}_capture_area"].unary_union

overlap_geom = tissue_gdf.intersection(capture_gdf)

overlap_area_mm2 = (overlap_geom.area * microns_per_pixel**2) / 1000000

print(f"Capture area covered by tissue: {overlap_area_mm2:.2f} mm²")

Capture area covered by tissue: 15.46 mm²


## Capture effeciency metrics

In [8]:
probe_set_df = pd.read_csv(f"{path_outs}/probe_set.csv", comment='#')
included = set(probe_set_df[probe_set_df['included'] == True]['gene_id'])
adata = sdata.tables['table'][:, sdata.tables['table'].var['gene_ids'].isin(included)]

In [9]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = len(adata.obs[adata.X.sum(axis=1) > 0])
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero}")

Number of spots in table with at least one UMI: 4904


In [10]:
# Number of spots in tissue with at least one UMI
in_tissue_mask = adata.obs['in_tissue_annotation'] == 1
umis_mask = np.array(adata.X.sum(axis=1)).flatten() > 0
spots_mask = in_tissue_mask & umis_mask
n_spots_in_tissue_not_zero = spots_mask.sum()
print(f"Number of spots in tissue with at least one UMI: {n_spots_in_tissue_not_zero}")

Number of spots in tissue with at least one UMI: 1779


In [28]:
# Total number of genes
print(f"Total number of genes: {(adata.X.sum(axis=0) > 0).sum()}")

Total number of genes: 19031


In [29]:
# Number of genes in tissue
print(f"Number of genes in tissue: {(adata[adata.obs['in_tissue_annotation'] == 1].X.sum(axis=0) > 0).sum()}")

Number of genes in tissue: 19030


In [30]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum()}")

Total number of unique UMIs: 32650590.0


In [31]:
# Number of unique UMIs in tissue
print(f"Number of unique UMIs in tissue: {adata[adata.obs['in_tissue_annotation'] == 1].X.sum()}")

Number of unique UMIs in tissue: 31293146.0


In [32]:
# Total number of UMIs

import h5py
import numpy as np

# Open the HDF5 file
with h5py.File(f"{path_outs}/molecule_info.h5", "r") as f:
    # Load the feature_id, feature_idx, and count datasets
    feature_id = f['/features/id'][:]
    feature_idx = f['/feature_idx'][:]
    count = f['/count'][:]

    # Decode feature_id entries
    feature_id = np.array([i.decode('ascii') for i in feature_id])
    
    # Find indices to keep
    included_feature_mask = np.isin(feature_id, list(included))
    included_feature_indices = np.where(included_feature_mask)[0]
    included_counts_mask = np.isin(feature_idx, included_feature_indices)
    
    # Sum counts for filtered indices
    included_counts = count[included_counts_mask]
    total_sum = np.sum(included_counts)
    total_count = len(included_counts)

    print("Total number of UMIs for included genes:", total_sum)
    print("Number of unique UMIs for included genes:", total_count)


Total number of UMIs for included genes: 73289820
Number of unique UMIs for included genes: 32650590


## QC

In [33]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
sdata = qc_metrics(
    sdata,
    'table',
    species
)

2025-08-25 17:36:30,990 - harpy.utils._io - WARNING - layer with name 'table' already exists. Overwriting...


In [34]:
# Plot QC metrics
for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:

    values = sdata.tables["table"].obs[qc_metric].to_numpy()
    vmin, vmax = np.percentile(values, [1, 99])
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
    
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"scanpy - {qc_metric}"

    sdata.pl.render_images(
        element=HE_image, scale=best_scale
    ).pl.render_labels(
        element=f"{dataset_id}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=qc_metric, scale=best_scale, table_name='table', cmap="viridis", norm=norm
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=True,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
    
    cbar = fig.axes[-1]
    cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
    cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(plots_path, f'scanpy_{qc_metric}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path)

INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\scanpy_n_genes_by_counts.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\scanpy_total_counts.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\scanpy_pct_counts_in_top_50_genes.png
INFO     Rasterizing image for faster rendering.                                                                   
Plot saved at: D:/2024-07-UTBenchmark-SpC/data/processed/Visium/SCA013/harpy\plots\scanpy_total_counts_mt.png
INFO     Rasterizing image for faster rendering.                                      

## Export unique UMI counts per gene

In [35]:
adata_in_tissue = adata[adata.obs['in_tissue_annotation'] == 1]

gene_counts = np.array(adata_in_tissue.X.sum(axis=0)).flatten()

df_gene_counts = pd.DataFrame({
    "gene": adata_in_tissue.var_names,
    "total_counts": gene_counts,
    "total_counts_corrected": gene_counts / overlap_area_mm2
})

df_gene_counts.to_csv(os.path.join(os.path.dirname(output_path), "gene_total_counts.csv"), index=False)

In [36]:
# for table_layer in [*sdata.tables]:
#     print(f'running: {table_layer}')

#     labels_layer=f"{dataset_id}_labels"
#     shapes_layer=dataset_id
    
#     plots_path_table = os.path.join(plots_path, table_layer)
#     os.makedirs(plots_path_table, exist_ok=True)
    
#     # Preprocess data
#     sdata = hp.tb.preprocess_transcriptomics(
#         sdata=sdata,
#         size_norm=False,
#         labels_layer=labels_layer,
#         table_layer=table_layer,
#         output_layer=f"{table_layer}_processed",
#         min_counts=10,
#         min_cells=5,
#         highly_variable_genes=True,
#         max_value_scale=10,
#         n_comps = 50,
#         update_shapes_layers=False,
#         overwrite=True
#     )
    
#     gc.collect()
    
#     # Leiden clustering
#     leiden_resolutions = [0.6]

#     for res in leiden_resolutions:
#         res_str = str(res).replace('.', '')
                
#         sdata = hp.tb.leiden(
#             sdata, 
#             labels_layer=labels_layer,
#             table_layer=f"{table_layer}_processed",
#             output_layer=f"{table_layer}_processed",
#             calculate_umap=True,
#             calculate_neighbors=True,
#             rank_genes=True,
#             n_neighbors=35,
#             n_pcs=30,
#             resolution=res,
#             key_added=f'leiden_{res_str}',
#             index_names_var=None,
#             index_positions_var=None,
#             random_state=100,
#             overwrite=True
#         )
        
#         hp.pl.cluster(
#             sdata, 
#             table_layer = f"{table_layer}_processed",
#             key_added=f'leiden_{res_str}',
#             output=os.path.join(plots_path_table, f'UMAP_leiden_{res_str}')
#         )
        
#         fig, ax = plt.subplots(figsize=figsize, dpi=300)
#         ax.set_xlabel("x", fontsize=12)
#         ax.set_ylabel("y", fontsize=12)

#         sdata.pl.render_images(
#             element=hne_img_layer
#         ).pl.render_labels(
#             element=labels_layer,
#             color=f'leiden_{res_str}',
#             table_name=f"{table_layer}_processed",
#             cmap="rainbow",
#             fill_alpha=1, 
#             contour_px=1, 
#             outline_alpha=0
#         ).pl.show(
#             coordinate_systems="global",
#             title=f"Leiden {res_str}",
#             figsize=figsize,
#             dpi=300,
#             colorbar=True,
#             ax=ax,
#         )

#         save_path = os.path.join(plots_path_lvl, f'leiden_{res_str}.png')
#         fig.savefig(save_path, bbox_inches="tight", facecolor="white", transparent=False)

#         plt.close(fig)

#         print("Plot saved at:", save_path)
    
#         gc.collect()